In [ ]:
# ============================================================
# [0] 🔑 K-Means 군집화 (Clustering) 시작!
# ============================================================
# 💡 비지도 학습(Unsupervised Learning)이란?
# - 레이블 없는 데이터에서 패턴을 찾는 학습
# - 군집화, 차원 축소, 이상치 탐지 등
# - 데이터의 "숨은 구조"를 발견!
# 
# 💡 K-Means 알고리즘:
# 1. k개의 중심점(centroid)을 무작위로 초기화
# 2. 각 샘플을 가장 가까운 중심점의 클러스터에 할당
# 3. 각 클러스터의 평균으로 중심점 업데이트
# 4. 변화 없을 때까지 2~3 반복
# 
# 📌 make_blobs로 인공 데이터 생성:
# - 5개의 가우시안 분포 중심 (blob_centers)
# - 각 분포의 표준편차 다름 (blob_std)
# - 2000개 샘플
# 
# 💡 fit_predict: 학습 + 클러스터 할당을 한번에
# - 결과는 0, 1, 2, 3, 4 같은 클러스터 인덱스 (실제 의미는 없음!)
# ============================================================

import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

blob_centers = np.array([
    [0.2, 2.3],
    [-1.5, 2.3],
    [-2.8, 1.8],
    [-2.8, 2.8],
    [-2.8, 1.3]
])

blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])

X, y = make_blobs(
    n_samples=2000,
    centers=blob_centers,
    cluster_std=blob_std,
    random_state=7
)

k = 5
kmeans = KMeans(n_clusters=k, random_state=42)
y_pred = kmeans.fit_predict(X)

In [ ]:
# [1] 💡 각 샘플의 클러스터 인덱스 확인
#      예: [4, 0, 1, ...] → 첫 샘플은 클러스터 4, 두번째는 0...

y_pred

In [ ]:
# [2] 💡 y_pred와 labels_는 같은 객체! (is 연산자로 확인)
#       fit_predict()는 사실 labels_를 반환

y_pred is kmeans.labels_

In [ ]:
# ============================================================
# [3] 🔑 cluster_centers_ - 학습된 클러스터 중심점들
# ============================================================
# - shape: (n_clusters, n_features) = (5, 2)
# - 각 행이 하나의 클러스터 중심
# - 이게 바로 K-Means가 찾은 "대표 위치"
# ============================================================

kmeans.cluster_centers_

In [ ]:
# [4] 💡 새 데이터를 가장 가까운 클러스터에 할당
#       - 새 점이 어느 클러스터에 속할지 예측
#       - 거리 계산으로 결정

import numpy as np
X_new = np.array([[0, 2], [3, 2], [-3, 3], [-3, 2.5]])
kmeans.predict(X_new)

In [ ]:
# ============================================================
# [5] 🔑 transform() - 각 클러스터 중심까지의 거리
# ============================================================
# 💡 K-Means는 분류기처럼 쓰는 게 아니라 변환기로도 쓸 수 있음!
# - 결과: (n_samples, n_clusters) 모양
# - 각 행: 해당 샘플이 각 클러스터 중심과 얼마나 떨어졌는지
# - 차원 축소 효과: n_features → n_clusters!
# 
# 응용: 클러스터 거리를 새 특성으로 사용 가능
# ============================================================

kmeans.transform(X_new).round(2)

In [ ]:
# ============================================================
# [6] 🔑 좋은 초기값으로 빠른 수렴
# ============================================================
# 💡 K-Means의 약점:
# - 초기 중심점 위치에 따라 다른 결과 나옴
# - 운 나쁘면 나쁜 지역 최솟값에 갇힘
# 
# 📌 해결책:
# - init=good_init: 직접 좋은 초기값 지정
# - n_init=1: 한 번만 시도 (기본은 여러 번 시도 후 최고 선택)
# 
# 💡 사이킷런 기본값:
# - init="k-means++" (똑똑한 초기화)
# - n_init=10: 10번 시도 후 가장 좋은 결과 선택
# ============================================================

good_init = np.array([[-3, 3], [-3, 2], [-3, 1], [-1, 2], [0, 2]])
kmeans = KMeans(n_clusters=5, init=good_init, n_init=1, random_state=42)
kmeans.fit(X)

In [ ]:
# ============================================================
# [7] 🔑 inertia_ - K-Means의 비용 함수
# ============================================================
# 💡 정의: 각 샘플과 가장 가까운 중심점 사이의 거리 제곱 합
# - 작을수록 좋음 (샘플들이 중심점에 가까이 모임)
# - K-Means는 이 값을 최소화하는 방향으로 학습
# 
# 💡 활용:
# - 모델 비교 (같은 k끼리)
# - 엘보 방법(elbow method)으로 k 결정
# ============================================================

kmeans.inertia_

In [ ]:
# [8] 💡 score()는 -inertia_ 반환 (사이킷런 관례: 클수록 좋음)
#       음수 부호: 거리는 작을수록 좋으니까 부호 뒤집기

kmeans.score(X)

In [ ]:
# ============================================================
# [9] 🔑 MiniBatchKMeans - 빅데이터용 K-Means
# ============================================================
# 💡 일반 KMeans vs MiniBatchKMeans:
# - 일반: 모든 데이터로 매 반복 (정확하지만 느림)
# - 미니배치: 일부 샘플로 반복 (빠르지만 약간 부정확)
# 
# 💡 언제 쓰나?
# - 데이터가 매우 크거나 메모리에 안 들어갈 때
# - 약간의 정확도를 희생하고 속도가 중요할 때
# - 사용법은 KMeans와 동일!
# ============================================================

from sklearn.cluster import MiniBatchKMeans

minibatch_kmeans = MiniBatchKMeans(n_clusters=5, random_state=42)
minibatch_kmeans.fit(X)

In [ ]:
# ============================================================
# [10] 🔑 엘보 방법(Elbow Method) - 최적 k 찾기
# ============================================================
# 💡 알고리즘:
# 1. k=1, 2, 3, ..., 9 각각 학습
# 2. 각 k에 대한 inertia_ 저장
# 3. k vs inertia_ 그래프 그리기
# 4. "팔꿈치(elbow)" 지점이 최적 k
# 
# 👁️ 해석:
# - k 증가 → inertia 감소 (당연)
# - 하지만 어느 지점부터 감소가 완만해짐
# - 그 "꺾이는 지점"이 최적 k!
# - 이 데이터에서는 k=4 부근이 엘보
# 
# 💡 한계: 엘보가 항상 명확하지 않음 → 실루엣 점수 병용
# ============================================================

import matplotlib.pyplot as plt

# 그림 9-8
kmeans_per_k = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
  for k in range(1, 10)]
inertias = [model.inertia_ for model in kmeans_per_k]

plt.figure(figsize=(8, 3.5))
plt.plot(range(1, 10), inertias, "bo-")
plt.xlabel("$k$")
plt.ylabel("inertia")
plt.annotate("", xy=(4, inertias[3]), xytext=(4.45, 650),
             arrowprops=dict(facecolor='black', shrink=0.1))
plt.text(4.5, 650, "elbow", horizontalalignment="center")
plt.axis([1, 8.5, 0, 1300])
plt.grid()
plt.show()

In [ ]:
# ============================================================
# [11] 🔑 실루엣 점수(Silhouette Score) - 더 정교한 k 평가
# ============================================================
# 💡 정의:
# - 각 샘플마다 실루엣 계수 계산: (b - a) / max(a, b)
#   - a: 같은 클러스터 내 평균 거리
#   - b: 가장 가까운 다른 클러스터의 평균 거리
# - 전체 평균이 실루엣 점수
# 
# 💡 해석:
# - 1에 가까울수록 좋음 (같은 클러스터끼리 가깝고 다른 클러스터와 멈)
# - 0에 가까우면 경계에 있음
# - 음수면 잘못된 클러스터에 할당됨!
# ============================================================

from sklearn.metrics import silhouette_score
silhouette_score(X, kmeans.labels_)

In [ ]:
# ============================================================
# [12] 💡 k에 따른 실루엣 점수 시각화
# ============================================================
# 👁️ 엘보 방법과 실루엣 점수 비교:
# - 엘보: 시각적으로 꺾이는 지점 (주관적)
# - 실루엣: 점수가 가장 높은 k (객관적)
# 
# 두 방법을 같이 보고 최적 k 결정!
# ============================================================

# 그림 9-9
silhouette_scores = [silhouette_score(X, model.labels_)
  for model in kmeans_per_k[1:]]
plt.figure(figsize=(8, 3))
plt.plot(range(2, 10), silhouette_scores, "bo-")
plt.xlabel("$k$")
plt.ylabel("Silhouette score")
plt.axis([1.8, 8.5, 0.55, 0.7])
plt.grid()
plt.show()

In [ ]:
# [13] 💡 K-Means 응용 1: 이미지 색상 분할 - 무당벌레 이미지 다운로드

import urllib.request
from pathlib import Path

homl3_root = "https://github.com/ageron/handson-ml3/raw/main/"
filename = "ladybug.png"
filepath = Path() / filename
if not filepath.is_file():
  print("Downloading", filename)
  url = f"{homl3_root}/images/unsupervised_learning/{filename}"
  urllib.request.urlretrieve(url, filepath)

In [ ]:
# [14] 💡 이미지를 NumPy 배열로 로드
#       shape: (height, width, 3) - RGB 3채널

import PIL
image = np.asarray(PIL.Image.open(filepath))
image.shape

In [ ]:
# ============================================================
# [15] 🔑 K-Means로 이미지 색상 분할(segmentation)
# ============================================================
# 💡 아이디어:
# - 이미지를 픽셀의 모음으로 봄 (각 픽셀은 RGB 3차원 점)
# - K-Means로 8가지 대표 색상 찾기
# - 각 픽셀을 가장 가까운 대표 색상으로 대체
# 
# 📌 핵심 코드:
# - X = image.reshape(-1, 3): 이미지 → 픽셀 리스트
# - cluster_centers_[labels_]: 각 픽셀을 클러스터 중심 색상으로
# - reshape: 다시 이미지 모양으로
# 
# 효과: 8가지 색상만 사용하는 색칠된 이미지! (포스터화 효과)
# ============================================================

X = image.reshape(-1, 3)
kmeans = KMeans(n_clusters=8, random_state=42).fit(X)
segmented_img = kmeans.cluster_centers_[kmeans.labels_]
segmented_img = segmented_img.reshape(image.shape)

In [ ]:
# ============================================================
# [16] 💡 색상 수에 따른 분할 비교 - 시각적 이해!
# ============================================================
# 👁️ 색상 수 변화에 따른 효과:
# - 10가지 색: 거의 원본과 비슷
# - 8가지: 약간 단순화
# - 6가지: 명확히 단순화
# - 4가지: 만화풍
# - 2가지: 흑백에 가까움
# 
# 💡 응용: 이미지 압축, 포스터 효과, 스타일 변환
# ============================================================

# 그림 9-12
segmented_imgs = []
n_colors = (10, 8, 6, 4, 2)
for n_clusters in n_colors:
  kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(X)
  segmented_img = kmeans.cluster_centers_[kmeans.labels_]
  segmented_imgs.append(segmented_img.reshape(image.shape))

plt.figure(figsize=(10, 5))
plt.subplots_adjust(wspace=0.05, hspace=0.1)

plt.subplot(2, 3, 1)
plt.imshow(image)
plt.title("OG")
plt.axis('off')

for idx, n_clusters in enumerate(n_colors):
  plt.subplot(2, 3, 2 + idx)
  plt.imshow(segmented_imgs[idx] / 255)
  plt.title(f"{n_clusters} colors")
  plt.axis('off')

plt.show()

In [ ]:
# ============================================================
# [17] 🎓 K-Means 응용 2: 준지도 학습(Semi-supervised Learning)
# ============================================================
# 💡 시나리오: 레이블이 매우 적을 때
# - 전체 1797개 손글씨 데이터
# - 훈련: 1400개, 테스트: 397개
# - 하지만 "레이블링 비용이 비싸서" 50개만 레이블 있다고 가정
# 
# 💡 준지도 학습 = 약간의 레이블 + 많은 비레이블 데이터
# - 실전에서 가장 흔한 상황!
# ============================================================

from sklearn.datasets import load_digits

X_digits, y_digits = load_digits(return_X_y=True)
X_train, y_train = X_digits[:1400], y_digits[:1400]
X_test, y_test = X_digits[1400:], y_digits[1400:]

In [ ]:
# [18] 💡 베이스라인: 50개 레이블만 사용해 학습

from sklearn.linear_model import LogisticRegression

n_labeled = 50
log_reg = LogisticRegression(max_iter=10_000)
log_reg.fit(X_train[:n_labeled], y_train[:n_labeled])

In [ ]:
# [19] 💡 베이스라인 정확도 - 보통 70~80% 정도
#       데이터가 50개로 너무 적어 성능 낮음

log_reg.score(X_test, y_test)

In [ ]:
# ============================================================
# [20] 🔑 K-Means로 "대표 샘플" 찾기
# ============================================================
# 💡 똑똑한 전략:
# 1. K-Means로 데이터를 50개 클러스터로 나눔
# 2. 각 클러스터에서 중심에 가장 가까운 샘플 = "대표 샘플"
# 3. 이 대표들만 레이블링하면 효율적!
# 
# 📌 핵심:
# - fit_transform: 학습 + 각 샘플의 클러스터 거리 반환
# - argmin(axis=0): 각 클러스터에서 가장 가까운 샘플 인덱스
# 
# 💡 일반적인 50개 무작위 샘플보다 훨씬 좋은 50개!
# ============================================================

k = 50
kmeans = KMeans(n_clusters=k, random_state=42)
X_digits_dist = kmeans.fit_transform(X_train)
representative_digit_idx = np.argmin(X_digits_dist, axis=0)
X_representative_digits = X_train[representative_digit_idx]

In [ ]:
# [21] 💡 대표 50개 숫자 시각화
#       다양한 모양의 0~9 숫자가 골고루 보임 ✅
#       무작위로 50개 뽑으면 일부 숫자가 빠질 수도 있음

# 그림 9-13
plt.figure(figsize=(8, 2))
for index, X_representative_digit in enumerate(X_representative_digits):
  plt.subplot(k // 10, 10, index + 1)
  plt.imshow(X_representative_digit.reshape(8, 8), cmap="binary",
             interpolation="bilinear")
  plt.axis('off')

plt.show()

In [ ]:
# [22] 💡 이제 이 대표 50개에 사람이 레이블링 (여기선 정답 사용)

y_representative_digits = y_train[representative_digit_idx]

In [ ]:
# ============================================================
# [23] ✅ 대표 샘플로 학습한 결과 - 베이스라인 대비 향상!
# ============================================================
# 👁️ 같은 50개 레이블이지만:
# - 무작위 50개 (셀 [19]): 약 75%
# - 대표 50개 (셀 [23]): 약 85%+
# 
# 💡 교훈: 레이블링 비용이 비쌀 때, 무엇을 레이블링할지 잘 선택!
# ============================================================

log_reg = LogisticRegression(max_iter=10_000)
log_reg.fit(X_representative_digits, y_representative_digits)
log_reg.score(X_test, y_test)

In [ ]:
# ============================================================
# [24] 🔑 레이블 전파(Label Propagation) - 한 걸음 더!
# ============================================================
# 💡 아이디어:
# - 같은 클러스터의 샘플은 같은 레이블일 것이라고 가정
# - 대표 샘플의 레이블을 클러스터 전체에 전파!
# 
# 결과: 1400개 모두 레이블 (단 대표 50개의 레이블을 복사)
# ============================================================

y_train_propagated = np.empty(len(X_train), dtype=np.int64)
for i in range(k):
  y_train_propagated[kmeans.labels_ == i] = y_representative_digits[i]

In [ ]:
# [25] 💡 전파된 1400개 레이블로 학습
#       데이터 양이 많아져 성능 더 향상!

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train_propagated)
log_reg.score(X_test, y_test)

In [ ]:
# ============================================================
# [26] 🔑 부분 전파(Partial Propagation) - 더 신중하게
# ============================================================
# 💡 문제점: 클러스터 경계 근처 샘플은 잘못된 레이블 가능성
# 
# 📌 해결책:
# - 각 클러스터에서 중심에 가까운 99%만 전파
# - 멀리 있는 1%(경계 근처)는 제외 → 노이즈 감소
# 
# percentile_closest=99: 가장 가까운 99% 사용
# ============================================================

percentile_closest = 99

X_cluster_dist = X_digits_dist[np.arange(len(X_train)), kmeans.labels_]
for i in range(k):
  in_cluster = (kmeans.labels_ == i)
  cluster_dist = X_cluster_dist[in_cluster]
  cutoff_distance = np.percentile(cluster_dist, percentile_closest)
  above_cutoff = (X_cluster_dist > cutoff_distance)
  X_cluster_dist[in_cluster & above_cutoff] = -1

partially_propagated = (X_cluster_dist != -1)
X_train_partially_propagated = X_train[partially_propagated]
y_train_partially_propagated = y_train_propagated[partially_propagated]

In [ ]:
# [27] 💡 부분 전파로 학습한 결과
#       더 정확한 레이블만 사용 → 성능 더 향상!

log_reg = LogisticRegression(max_iter=10_000)
log_reg.fit(X_train_partially_propagated, y_train_partially_propagated)
log_reg.score(X_test, y_test)

In [ ]:
# [28] 💡 전파된 레이블의 정확도 확인
#       대부분 진짜 레이블과 일치 ✅

(y_train_partially_propagated == y_train[partially_propagated]).mean()

In [ ]:
# ============================================================
# [29] 🔑 DBSCAN - 밀도 기반 군집화
# ============================================================
# 💡 K-Means의 한계:
# - 클러스터 수 k를 미리 지정해야 함
# - 동그란 모양의 클러스터만 잘 찾음
# - 이상치(outlier)에 약함
# 
# 💡 DBSCAN의 장점:
# - 클러스터 수 자동 결정
# - 어떤 모양이든 OK (반달, 나선 등)
# - 이상치를 -1로 따로 분류
# 
# 📌 매개변수:
# - eps=0.05: "이웃" 거리 (작을수록 빡빡한 클러스터)
# - min_samples=5: 코어 포인트 되려면 필요한 이웃 수
# 
# 💡 데이터: make_moons - 반달 두 개 (K-Means로는 잘 못 나눔!)
# ============================================================

from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.05)
dbscan = DBSCAN(eps=0.05, min_samples=5)
dbscan.fit(X)

In [ ]:
# [30] 💡 DBSCAN 결과 확인
#       0, 1, 2, ... 클러스터 번호 + -1 (이상치)

dbscan.labels_

In [ ]:
# [31] core_sample_indices_의 길이 = 코어 포인트 개수

len(dbscan.core_sample_indices_)

In [ ]:
# [32] 💡 core_sample_indices_ : 코어 포인트들의 인덱스
#       코어 포인트 = 주변에 충분한 이웃이 있는 "확실한" 샘플

dbscan.core_sample_indices_

In [ ]:
# [33] 💡 components_ : 코어 포인트들의 실제 데이터

dbscan.components_

In [ ]:
# ============================================================
# [34] ⚠️ DBSCAN은 predict() 메서드가 없음!
# ============================================================
# 💡 해결책: KNeighborsClassifier로 대체
# - DBSCAN으로 클러스터 학습
# - 코어 포인트들로 KNN 학습
# - 새 데이터는 KNN으로 분류
# 
# 💡 이게 비지도→지도 변환의 좋은 예!
# ============================================================

from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=50)
knn.fit(dbscan.components_, dbscan.labels_[dbscan.core_sample_indices_])

In [ ]:
# [35] 새 샘플에 대한 KNN 예측

X_new = np.array([[-0.5, 0], [0, 0.5], [1, -0.1], [2, 1]])
knn.predict(X_new)

In [ ]:
# [36] 💡 각 클래스 확률

knn.predict_proba(X_new)

In [ ]:
# ============================================================
# [37] 🔑 이상치 처리 - 가까운 클러스터가 없으면 이상치로
# ============================================================
# - kneighbors: 가장 가까운 이웃과의 거리 반환
# - y_dist > 0.2: 거리가 0.2 초과 = 너무 멈
# - 그런 샘플은 -1 (이상치)로 표시
# 
# 💡 응용: 사기 탐지, 결함 검출 등
# ============================================================

y_dist, y_pred_idx = knn.kneighbors(X_new, n_neighbors=1)
y_pred = dbscan.labels_[dbscan.core_sample_indices_][y_pred_idx]
y_pred[y_dist > 0.2] = -1
y_pred.ravel()

In [ ]:
# ============================================================
# [38] 🔑 가우시안 혼합 모델(GMM) - 확률적 군집화
# ============================================================
# 💡 데이터 준비: 일부러 복잡한 모양 만들기
# - X1: 두 클러스터를 변환행렬로 회전/늘림 (타원형)
# - X2: 동그란 클러스터 하나 추가
# - 총 3개의 "다른 모양/크기" 클러스터
# 
# K-Means는 동그란 클러스터만 잘 처리 → GMM 필요!
# ============================================================

X1, y1 = make_blobs(n_samples=1000, centers=((4, -4), (0, 0)),
random_state=42)
X1 = X1.dot(np.array([[0.374, 0.95], [0.732, 0.598]]))
X2, y2 = make_blobs(n_samples=250, centers=1, random_state=42)
X2 = X2 + [6, -8]
X = np.r_[X1, X2]
Y = np.r_[y1, y2]

In [ ]:
# [39] 💡 데이터 시각화 - 다양한 모양의 클러스터 확인

import matplotlib.pyplot as plt

def plot_clusters(X, y=None):
    plt.scatter(X[:, 0], X[:, 1], c=y, s=1)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$", rotation=0)

plt.figure(figsize=(8, 4))
plot_clusters(X)
plt.gca().set_axisbelow(True)
plt.grid()
plt.show()

In [ ]:
# ============================================================
# [40] 🔑 GaussianMixture (가우시안 혼합 모델)
# ============================================================
# 💡 GMM이란?
# - "데이터가 여러 가우시안 분포의 혼합으로 생성됐다"고 가정
# - 각 가우시안 = 하나의 클러스터 (타원형 가능!)
# - K-Means보다 유연한 모델
# 
# 📌 매개변수:
# - n_components=3: 3개의 가우시안
# - n_init=10: 10번 시도 후 최고 선택
# 
# 💡 학습 알고리즘: EM(Expectation-Maximization)
# - E단계: 각 샘플의 클러스터 소속 확률 계산
# - M단계: 각 가우시안의 파라미터 업데이트
# - 반복!
# ============================================================

from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=3, n_init=10, random_state=42)
gm.fit(X)

In [ ]:
# ============================================================
# [41] 💡 학습된 가우시안 혼합 비율
# ============================================================
# - weights_: 각 클러스터의 비중 (합은 1)
# - 예: [0.4, 0.3, 0.3] → 첫 클러스터가 40%
# ============================================================

gm.weights_

In [ ]:
# [42] 💡 means_: 각 가우시안의 평균 (= 중심점)
#       K-Means의 cluster_centers_와 비슷한 역할

gm.means_

In [ ]:
# ============================================================
# [43] 💡 covariances_: 공분산 행렬 - GMM의 핵심!
# ============================================================
# 💡 K-Means와 결정적 차이:
# - K-Means: 동그란 클러스터만 (단일 거리)
# - GMM: 공분산으로 "타원형" 표현 가능
# 
# - shape: (n_components, n_features, n_features)
# - 각 가우시안마다 하나의 공분산 행렬
# - 대각선: 각 축의 분산
# - 비대각선: 축 간 상관관계 (회전된 타원)
# ============================================================

gm.covariances_

In [ ]:
# [44] 💡 converged_: 알고리즘이 수렴했는지 (True가 좋음)

gm.converged_

In [ ]:
# [45] 💡 n_iter_: 수렴까지 걸린 반복 횟수

gm.n_iter_

In [ ]:
# [46] 💡 predict(): 각 샘플의 클러스터 (가장 가능성 높은)
#       하드 클러스터링

gm.predict(X)

In [ ]:
# ============================================================
# [47] 🔑 predict_proba() - GMM의 강력한 기능!
# ============================================================
# 💡 K-Means에는 없는 기능: "소프트 클러스터링"
# - 각 샘플이 각 클러스터에 속할 확률 반환
# - 예: [0.95, 0.04, 0.01] → 95% 클러스터 0, 4% 클러스터 1, 1% 클러스터 2
# 
# 💡 활용:
# - 경계에 있는 모호한 샘플 식별
# - 확신도 기반 의사결정
# ============================================================

gm.predict_proba(X).round(3)

In [ ]:
# ============================================================
# [48] 🔑 sample() - GMM은 생성 모델!
# ============================================================
# 💡 학습된 GMM에서 새로운 샘플을 "생성"할 수 있음
# - 분포를 학습했으니 그 분포에서 샘플링 가능
# - K-Means는 못 함!
# 
# 💡 생성 모델의 응용:
# - 데이터 증강
# - 가상 데이터 생성
# - GAN, VAE의 전신 개념
# ============================================================

X_new, y_new = gm.sample(6)

In [ ]:
# [49] 생성된 6개 샘플 데이터

X_new

In [ ]:
# [50] 생성된 샘플이 어느 클러스터에서 왔는지

y_new

In [ ]:
# ============================================================
# [51] 🔑 score_samples() - 각 샘플의 로그 밀도
# ============================================================
# 💡 정의: 해당 위치에서의 확률 밀도(log)
# - 값이 클수록 데이터의 "정상" 위치
# - 값이 작을수록 데이터에서 멀리 떨어진 "이상" 위치
# 
# 💡 활용: 이상치 탐지의 기반!
# ============================================================

gm.score_samples(X).round(2)

In [ ]:
# ============================================================
# [52] 🔑 GMM으로 이상치 탐지!
# ============================================================
# 💡 알고리즘:
# 1. 모든 샘플의 밀도 계산
# 2. 하위 2%를 임계값으로 (np.percentile)
# 3. 임계값 미만은 이상치!
# 
# 💡 K-Means보다 훨씬 정교한 이상치 탐지:
# - 클러스터 모양 고려
# - 밀도 기반 판단
# ============================================================

densities = gm.score_samples(X)
density_threshold = np.percentile(densities, 2)
anomalies = X[densities < density_threshold]

In [ ]:
# ============================================================
# [53] 🔑 BIC (Bayesian Information Criterion) - 모델 선택 지표
# ============================================================
# 💡 정의: 모델의 좋음을 측정 (작을수록 좋음)
# - 우도(likelihood) + 모델 복잡도 페널티
# - 너무 많은 컴포넌트는 페널티
# 
# 💡 활용: 최적 n_components 찾기!
# - 여러 n_components로 학습 후 BIC 최소인 것 선택
# ============================================================

gm.bic(X)

In [ ]:
# ============================================================
# [54] 💡 AIC (Akaike Information Criterion) - BIC와 유사
# ============================================================
# - BIC와 비슷하지만 페널티 다름
# - AIC는 BIC보다 큰 모델 선호 경향
# - 둘 다 작을수록 좋음
# ============================================================

gm.aic(X)

In [ ]:
# ============================================================
# [55] 🔑 BayesianGaussianMixture - 자동으로 k 결정!
# ============================================================
# 💡 일반 GMM의 단점: n_components 미리 정해야 함
# 
# 💡 베이지안 GMM의 마법:
# - n_components를 크게 잡아도 OK (10으로 설정)
# - 불필요한 컴포넌트는 자동으로 가중치 0에 가깝게
# - 결과적으로 적절한 클러스터 수 자동 발견
# 
# 👁️ weights_를 보면 일부는 0.4 같은 큰 값,
#     일부는 0.001 같은 매우 작은 값 → 후자는 무시!
# ============================================================

from sklearn.mixture import BayesianGaussianMixture
bgm = BayesianGaussianMixture(n_components=10, n_init=10, random_state=42)
bgm.fit(X)
bgm.weights_.round(2)

In [ ]:
# ============================================================
# [56] 🎓 종합 실습: 올리베티 얼굴 데이터셋
# ============================================================
# 💡 데이터: 40명의 사람, 각 10장 = 총 400장 얼굴
# - 64x64 흑백 이미지 (4096 픽셀)
# - 같은 사람이 다양한 표정/조명으로 촬영
# 
# 목표: 클러스터링으로 같은 사람끼리 묶어보기
# ============================================================

from sklearn.datasets import fetch_olivetti_faces

olivetti = fetch_olivetti_faces()

In [ ]:
# [57] 데이터셋 설명 출력

print(olivetti.DESCR)

In [ ]:
# [58] target 확인 - 각 사진의 사람 ID (0~39)

olivetti.target

In [ ]:
# ============================================================
# [59] 🔑 계층적 샘플링으로 훈련/검증/테스트 분할
# ============================================================
# 💡 StratifiedShuffleSplit:
# - 각 클래스(사람) 비율을 유지하며 분할
# - 사람마다 사진이 골고루 분배되도록
# 
# 📌 2단계 분할:
# 1. 전체 → 훈련+검증 / 테스트
# 2. 훈련+검증 → 훈련 / 검증
# 
# n_splits=1: 한 번만 분할 (교차 검증 아님)
# ============================================================

from sklearn.model_selection import StratifiedShuffleSplit

strat_split = StratifiedShuffleSplit(n_splits=1, test_size=40, random_state=42)
train_valid_idx, test_idx = next(strat_split.split(olivetti.data,
                                                   olivetti.target))

X_train_valid = olivetti.data[train_valid_idx]
y_train_valid = olivetti.target[train_valid_idx]
X_test = olivetti.data[test_idx]
y_test = olivetti.target[test_idx]

strat_split = StratifiedShuffleSplit(n_splits=1, test_size=80, random_state=43)
train_idx, valid_idx = next(strat_split.split(X_train_valid, y_train_valid))
X_train = X_train_valid[train_idx]
y_train = y_train_valid[train_idx]
X_valid = X_train_valid[valid_idx]
y_valid = y_train_valid[valid_idx]

In [ ]:
# [60] 각 세트의 shape 확인

print(X_train.shape, y_train.shape)
print(X_valid.shape, y_valid.shape)
print(X_test.shape, y_test.shape)

In [ ]:
# ============================================================
# [61] 🔑 PCA로 차원 축소 (전처리)
# ============================================================
# 💡 왜 PCA를 먼저?
# - 원본 4096차원은 너무 큼 → K-Means 느림
# - 99% 분산 보존 → 거의 정보 손실 없음
# - 보통 100차원 정도로 줄어듦
# 
# ⭐ 8장 배운 PCA를 9장 K-Means 전처리에 활용!
# ============================================================

from sklearn.decomposition import PCA

pca = PCA(0.99)
X_train_pca = pca.fit_transform(X_train)
X_valid_pca = pca.transform(X_valid)
X_test_pca = pca.transform(X_test)

pca.n_components_

In [ ]:
# ============================================================
# [62] 💡 다양한 k로 K-Means 학습 (시간 좀 걸림)
# ============================================================
# - k_range: 5부터 145까지 5씩 증가 (29가지)
# - 각 k로 학습 후 리스트에 저장
# - 진짜 사람 수는 40명이지만 K-Means가 알아낼지 확인!
# ============================================================

from sklearn.cluster import KMeans
k_range = range(5, 150, 5)
kmeans_per_k = []
for k in k_range:
  print(f"k={k}")
  kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
  kmeans.fit(X_train_pca)
  kmeans_per_k.append(kmeans)

In [ ]:
# ============================================================
# [63] 💡 실루엣 점수로 최적 k 찾기
# ============================================================
# 👁️ 그래프에서 빨간 사각형 = 최적 k
# - 보통 사람 수(40)에 가까운 값이 나옴
# - 100~120 정도가 최적일 수도 (사람당 여러 표정 그룹 형성)
# ============================================================

from sklearn.metrics import silhouette_score

silhouette_scores = [silhouette_score(X_train_pca, model.labels_)
for model in kmeans_per_k]
best_index = np.argmax(silhouette_scores)
best_k = k_range[best_index]
best_score = silhouette_scores[best_index]

plt.figure(figsize=(8, 3))
plt.plot(k_range, silhouette_scores, "bo-")
plt.xlabel("$k$")
plt.ylabel("Silhouette score")
plt.plot(best_k, best_score, "rs")
plt.grid()
plt.show()

In [ ]:
# [64] 💡 최적 k의 모델 선택

best_model = kmeans_per_k[best_index]

In [ ]:
# ============================================================
# [65] 🎯 클러스터별 얼굴 시각화 - 비지도 학습의 위력!
# ============================================================
# 💡 각 클러스터의 얼굴들을 모아서 시각화
# - 같은 사람끼리 잘 모였는지 눈으로 확인
# - 다양한 사람이 섞였다면 K-Means 한계
# - 같은 사람만 모였다면 성공! ✅
# 
# 👁️ 관찰:
# - 어떤 클러스터는 같은 사람 (성공)
# - 어떤 클러스터는 다른 사람 섞임 (조명/표정이 비슷해서)
# 
# 💡 이게 비지도 학습의 한계와 가능성을 보여줌
# ============================================================

def plot_faces(faces, labels, n_cols=5):
  faces = faces.reshape(-1, 64, 64)
  n_rows = (len(faces) - 1) // n_cols + 1
  plt.figure(figsize=(n_cols, n_rows * 1.1))
  for index, (face, label) in enumerate(zip(faces, labels)):
    plt.subplot(n_rows, n_cols, index + 1)
    plt.imshow(face, cmap="gray")
    plt.axis("off")
    plt.title(label)
  plt.show()

for cluster_id in np.unique(best_model.labels_):
  print("Cluster", cluster_id)
  in_cluster = best_model.labels_==cluster_id
  faces = X_train[in_cluster]
  labels = y_train[in_cluster]
  plot_faces(faces, labels)